In [ ]:
import sys
from pathlib import Path
import pandas as pd
import psycopg2

root_path = Path.cwd().resolve().parents[1]
if str(root_path) not in sys.path:
    sys.path.append(str(root_path))

from main import db_host, db_name, db_password, db_port, db_schema, db_user

with psycopg2.connect(
    host=db_host,
    dbname=db_name,
    user=db_user,
    password=db_password,
    port=db_port,
    sslmode="require"
) as conn:
    df_bronze = pd.read_sql(f"SELECT * FROM {db_schema}.bronze_deputados;", conn)

# display(df_bronze.head())

C:\Users\mauro.silva\AppData\Local\Temp\ipykernel_12804\2407781213.py:20: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_bronze = pd.read_sql(f"SELECT * FROM {db_schema}.bronze_deputados;", conn)


,n_id_deputado,s_ds_uri,s_ds_nome,s_cd_sigla_partido,s_ds_uri_partido,s_cd_sigla_uf,n_id_legislatura,s_ds_url_foto,s_ds_email,d_dh_carga
0,204379,https://dadosabertos.camara.leg.br/api/v2/depu...,Acácio Favacho,MDB,https://dadosabertos.camara.leg.br/api/v2/part...,AP,57,https://www.camara.leg.br/internet/deputado/ba...,dep.acaciofavacho@camara.leg.br,2026-11-26 10:36:43.438276
1,220714,https://dadosabertos.camara.leg.br/api/v2/depu...,Adail Filho,MDB,https://dadosabertos.camara.leg.br/api/v2/part...,AM,57,https://www.camara.leg.br/internet/deputado/ba...,dep.adailfilho@camara.leg.br,2026-11-26 10:36:43.438276
2,221328,https://dadosabertos.camara.leg.br/api/v2/depu...,Adilson Barroso,PL,https://dadosabertos.camara.leg.br/api/v2/part...,SP,57,https://www.camara.leg.br/internet/deputado/ba...,dep.adilsonbarroso@camara.leg.br,2026-11-26 10:36:43.438276
3,204560,https://dadosabertos.camara.leg.br/api/v2/depu...,Adolfo Viana,PSDB,https://dadosabertos.camara.leg.br/api/v2/part...,BA,57,https://www.camara.leg.br/internet/deputado/ba...,dep.adolfoviana@camara.leg.br,2026-11-26 10:36:43.438276
4,204528,https://dadosabertos.camara.leg.br/api/v2/depu...,Adriana Ventura,NOVO,https://dadosabertos.camara.leg.br/api/v2/part...,SP,57,https://www.camara.leg.br/internet/deputado/ba...,dep.adrianaventura@camara.leg.br,2026-11-26 10:36:43.438276


In [5]:
import sys
from pathlib import Path
from datetime import datetime
import pandas as pd
import psycopg2

# 1. Garante que a raiz do projeto esteja no sys.path e importa as credenciais
root_path = Path.cwd().resolve().parents[1]
if str(root_path) not in sys.path:
    sys.path.append(str(root_path))

from main import db_host, db_name, db_password, db_port, db_schema, db_user

# 2. Lê os dados da tabela Bronze
query_bronze = f"""
SELECT 
    n_id_deputado,
    s_ds_uri,
    s_ds_nome,
    s_cd_sigla_partido,
    s_ds_uri_partido,
    s_cd_sigla_uf,
    n_id_legislatura,
    s_ds_url_foto,
    s_ds_email,
    d_dh_carga
FROM {db_schema}.bronze_deputados;
"""

with psycopg2.connect(
    host=db_host,
    dbname=db_name,
    user=db_user,
    password=db_password,
    port=db_port,
    sslmode="require"
) as conn:
    df_silver = pd.read_sql(query_bronze, conn)

# 3. Transformações e padronizações da camada Silver
df_silver["n_id_deputado"] = df_silver["n_id_deputado"].astype(int)
df_silver["n_id_legislatura"] = df_silver["n_id_legislatura"].astype(int)

# Limpeza de strings e padronização de caixa
df_silver["s_ds_nome"] = df_silver["s_ds_nome"].str.strip()
df_silver["s_cd_sigla_partido"] = df_silver["s_cd_sigla_partido"].str.strip().str.upper()
df_silver["s_cd_sigla_uf"] = df_silver["s_cd_sigla_uf"].str.strip().str.upper()

# Trata e-mails vazios ou nulos e padroniza em minúsculo
df_silver["s_ds_email"] = (
    df_silver["s_ds_email"]
    .fillna("")
    .str.strip()
    .str.lower()
    .replace("", None)
)

# Remove eventuais duplicidades por ID mantendo o registro mais recente
df_silver = (
    df_silver.sort_values("d_dh_carga", ascending=False)
    .drop_duplicates(subset=["n_id_deputado"], keep="first")
    .reset_index(drop=True)
)

# Atualiza o timestamp de processamento da Silver
df_silver["d_dh_carga"] = datetime.now()

print(f"Registros carregados e tratados para Silver: {len(df_silver)}")
display(df_silver.head(10))

Registros carregados e tratados para Silver: 513


C:\Users\mauro.silva\AppData\Local\Temp\ipykernel_12804\2575326144.py:38: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_silver = pd.read_sql(query_bronze, conn)


,n_id_deputado,s_ds_uri,s_ds_nome,s_cd_sigla_partido,s_ds_uri_partido,s_cd_sigla_uf,n_id_legislatura,s_ds_url_foto,s_ds_email,d_dh_carga
0,204379,https://dadosabertos.camara.leg.br/api/v2/depu...,Acácio Favacho,MDB,https://dadosabertos.camara.leg.br/api/v2/part...,AP,57,https://www.camara.leg.br/internet/deputado/ba...,dep.acaciofavacho@camara.leg.br,2026-11-26 10:44:12.279071
1,178910,https://dadosabertos.camara.leg.br/api/v2/depu...,Joaquim Passarinho,PL,https://dadosabertos.camara.leg.br/api/v2/part...,PA,57,https://www.camara.leg.br/internet/deputado/ba...,dep.joaquimpassarinho@camara.leg.br,2026-11-26 10:44:12.279071
2,220647,https://dadosabertos.camara.leg.br/api/v2/depu...,Mauricio Neves,PP,https://dadosabertos.camara.leg.br/api/v2/part...,SP,57,https://www.camara.leg.br/internet/deputado/ba...,dep.mauricioneves@camara.leg.br,2026-11-26 10:44:12.279071
3,220550,https://dadosabertos.camara.leg.br/api/v2/depu...,Mauricio Marcon,PL,https://dadosabertos.camara.leg.br/api/v2/part...,RS,57,https://www.camara.leg.br/internet/deputado/ba...,dep.mauriciomarcon@camara.leg.br,2026-11-26 10:44:12.279071
4,220628,https://dadosabertos.camara.leg.br/api/v2/depu...,Mauricio do Vôlei,PL,https://dadosabertos.camara.leg.br/api/v2/part...,MG,57,https://www.camara.leg.br/internet/deputado/ba...,dep.mauriciodovolei@camara.leg.br,2026-11-26 10:44:12.279071
5,220609,https://dadosabertos.camara.leg.br/api/v2/depu...,Maurício Carvalho,UNIÃO,https://dadosabertos.camara.leg.br/api/v2/part...,RO,57,https://www.camara.leg.br/internet/deputado/ba...,dep.mauriciocarvalho@camara.leg.br,2026-11-26 10:44:12.279071
6,220661,https://dadosabertos.camara.leg.br/api/v2/depu...,Matheus Noronha,PL,https://dadosabertos.camara.leg.br/api/v2/part...,CE,57,https://www.camara.leg.br/internet/deputado/ba...,dep.matheusnoronha@camara.leg.br,2026-11-26 10:44:12.279071
7,178843,https://dadosabertos.camara.leg.br/api/v2/depu...,Marx Beltrão,UNIÃO,https://dadosabertos.camara.leg.br/api/v2/part...,AL,57,https://www.camara.leg.br/internet/deputado/ba...,dep.marxbeltrao@camara.leg.br,2026-11-26 10:44:12.279071
8,220572,https://dadosabertos.camara.leg.br/api/v2/depu...,Marussa Boldrin,REPUBLICANOS,https://dadosabertos.camara.leg.br/api/v2/part...,GO,57,https://www.camara.leg.br/internet/deputado/ba...,dep.marussaboldrin@camara.leg.br,2026-11-26 10:44:12.279071
9,204566,https://dadosabertos.camara.leg.br/api/v2/depu...,Marreca Filho,PODE,https://dadosabertos.camara.leg.br/api/v2/part...,MA,57,https://www.camara.leg.br/internet/deputado/ba...,dep.marrecafilho@camara.leg.br,2026-11-26 10:44:12.279071
